In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import re
import pandas as pd
import os
import matplotlib.pyplot as plt

# 2. Text Analysis

1. Topic Modeling: Apply algorithms like Latent Dirichlet Allocation (LDA) on Keywords, topics, and broad_topics to discover hidden thematic structures.
2. Term Frequency-Inverse Document Frequency (TF-IDF): Determine the importance of keywords in the context of the entire dataset.

This study aims to perform text mining and topic modeling on academic publications to identify key areas of focus, trends, and emerging research themes. Two popular techniques are utilized: Term Frequency-Inverse Document Frequency (TF-IDF) for evaluating the importance of keywords and Latent Dirichlet Allocation (LDA) for extracting latent topics from the corpus.

2.1 Term Frequency-Inverse Document Frequency (TF-IDF)

TF-IDF is a statistical measure used to evaluate the importance of a word in a document relative to a corpus. It is computed as:

Where:

 1. is the term frequency of term  in document .

 2. is the inverse document frequency of term  across the entire corpus , calculated as:

Here,  is the total number of documents in the corpus, and  is the number of documents containing term . The use of TF-IDF helps prioritize terms that are frequent within individual documents but rare across the entire dataset, highlighting important keywords and helping to identify key research areas.

2.2 Latent Dirichlet Allocation (LDA)

LDA is an unsupervised topic modeling technique that helps discover hidden thematic structures in a corpus by grouping words that commonly co-occur. LDA assumes that documents are a mixture of topics, where each topic is represented by a distribution over words. The model generates topics by finding patterns of word co-occurrence, which enables the identification of themes in the document collection. The output of LDA consists of topics, each represented by a list of significant keywords. The LDA algorithm was chosen for this study to explore thematic trends and classify the research papers by topics, thereby gaining insight into the latent subjects present in the corpus.



In [11]:
folder_path = '../../tcm/pe_outputs/1after_ground_truth'
df = pd.read_excel(os.path.join(folder_path, 'topics_citation_full_dataset_g.xlsx'))
df

,Index,File_Name,Paper_Name,Keywords,Topics,Journal_Name,Published_Year,Country,Continent,URL,...,1987,1986,1985,1984,1983,1982,1981,1980,topics,ground_truth_topics
0,0,(ASCE)0733-9364(1986)112_3(346).pdf,RESOURCE MANAGEMENT IN CONSTRUCTION,[],"['Resource management', 'Construction industry...",Journal of construction engineering and manage...,1986,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],"['Resource planning', 'cost control', 'constru..."
1,2,(ASCE)0742-597X(2005)21_1(2).pdf,Competency-Based Model for Predicting Construc...,['Human factors; Professional development; Pro...,"['Construction project management', 'Competenc...",Journal of Management in Engineering,2005,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],"['Competency-based performance prediction', 'c..."
2,3,(ASCE)0887-3801(2006)20_3(165).pdf,Multi-Agent Framework for General-Purpose Situ...,['Models; Simulation; Construction management;...,"['Multi-agent framework', 'Situational simulat...",Journal of Computing in Civil Engineering,2006,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],"['Multi-agent systems', 'situational simulatio..."
3,4,(ASCE)1532-6748(2001)1_2(17).pdf,Construction Management Practices Are Slowly C...,[],NaN,Leadership and Management in Engineering,2001,NaN,4,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],['Strategic management in construction industr...
4,5,(ASCE)CO.1943-7862.0000100.pdf,Managerial Competencies of Female and Male Con...,['Women; Discrimination; Workplace diversity; ...,"['Female project managers', 'Managerial compet...",Journal of Construction Engineering and Manage...,2009,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],"['Gender representation in construction', 'Man..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,743,Texto.11.ConstructionQuality.pdf,Construction Quality Management: Principles an...,"['Manufacturing', 'Marketing', 'R&D and Engine...",NaN,NaN,2012,NaN,743,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],"['Quality Management', 'Total Quality Manageme..."
648,744,Understanding the early stages of the innovati...,Understanding the early stages of the innovati...,"['Communication', 'critical perspective', 'dif...","['awareness', 'influence', 'communication netw...",Construction Management and Economics,2011,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],"['Innovation diffusion', 'awareness', 'influen..."
649,745,Use of attitude congruence to identify safety ...,Use of attitude congruence to identify safety ...,"['Safety', 'small business.']","['Attitude congruence', 'Safety interventions'...",Construction Management and Economics,2011,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],"['Construction industry safety', 'Occupational..."
650,746,Valuation of the minimum revenue guarantee and...,Valuation of the minimum revenue guarantee and...,"['BOT', 'infrastructure', 'real option', 'opti...","['Valuation', 'Minimum revenue guarantee', 'Op...",Construction Management and Economics,2006,China,Asia,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,0,[],"['Build-Operate-Transfer', 'minimum revenue gu..."


In [3]:
# Remove rows where 'Published_Year' has NA/missing values
# df = df.dropna(subset=['Published_Year'])
# df.loc[:, 'Published_Year'] = df['Published_Year'].astype(int)
# # Update the dataframe to replace NaN values in year columns (1980-2023) with 0 and filter out rows with NaN in 'Published_Year'
# year_columns = [col for col in df.columns if isinstance(col, int) and 1980 <= col <= 2023]
# df[year_columns] = df[year_columns].fillna(0)

In [4]:
# Replace NaN values in 'topics' and 'broad_topics' with empty lists for the convenience of further analysis
# df['topics'] = df['topics'].apply(lambda x: x if isinstance(x, list) else [])

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [13]:
def apply_lda(text_data_str, n_topics=5, n_top_words=10, use_ngrams=False):
    """
    text_data_str: A single string representing the entire document.
    n_topics:      Number of LDA topics.
    n_top_words:   Number of top words to extract from each topic.
    use_ngrams:    Whether to use bigrams (2-grams) in addition to unigrams.
    """
    # Convert the single string into a list with one element
    # (TfidfVectorizer expects an iterable of documents)
    if not text_data_str or not text_data_str.strip():
        # Handle the case where text might be empty or whitespace
        return []

    docs = [text_data_str]

    # Apply TF-IDF Vectorizer with optional n-grams
    ngram_range = (1, 2) if use_ngrams else (1, 1)
    tfidf_vectorizer = TfidfVectorizer(
        stop_words='english',
        max_features=1000,
        ngram_range=ngram_range
    )
    tfidf_matrix = tfidf_vectorizer.fit_transform(docs)

    # Fit LDA Model for Topic Modeling
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
    lda.fit(tfidf_matrix)

    # Get the vocabulary of terms
    feature_names = tfidf_vectorizer.get_feature_names_out()

    # Collect the top words for each topic
    all_topic_words = []
    for topic_idx, topic in enumerate(lda.components_):
        # Indices of top n_top_words
        top_word_indices = topic.argsort()[:-n_top_words - 1:-1]
        # Convert indices to actual words
        top_words = [feature_names[i] for i in top_word_indices]
        all_topic_words.append(top_words)

        # (Optional) Print the topic/words
        print(f"Topic #{topic_idx + 1}: {', '.join(top_words)}")

    return all_topic_words


# --------------------------------------------------------
# Example: Updating df['lda_topics']
# --------------------------------------------------------
def populate_lda_topics(df, folder_path, n_topics=5, n_top_words=10, use_ngrams=True):
    """
    Reads each paper in `folder_path` by matching the row's Index column,
    performs LDA, collects all topic words into a list, and stores it
    in df['lda_topics'] for each row.
    """

    # Create an empty column to store the results if not present
    if 'lda_topics' not in df.columns:
        df['lda_topics'] = None

    for i, row in df.iterrows():
        paper_index = row['Index']  # or whichever column holds the txt filename
        file_name = f"{paper_index}.txt"
        file_path = os.path.join(folder_path, file_name)

        if not os.path.exists(file_path):
            print(f"File not found for index: {paper_index} -> {file_path}")
            df.at[i, 'lda_topics'] = []
            continue

        # Read file
        with open(file_path, 'r', encoding='utf-8') as f:
            paper_content = f.read()

        # Apply LDA on this single paper
        lda_result = apply_lda(
            text_data_str=paper_content,
            n_topics=n_topics,
            n_top_words=n_top_words,
            use_ngrams=use_ngrams
        )

        # Post process: collect all words in a set, then convert to list
        unique_words = set()
        for topic_word_list in lda_result:
            unique_words.update(topic_word_list)
        unique_words_list = list(unique_words)

        # Store back into the DataFrame
        df.at[i, 'lda_topics'] = unique_words_list

    return df

In [14]:
# --------------------------------------------------------
# Usage Example
# --------------------------------------------------------
# df must have a column "Index" matching the filename (e.g., "1.txt", "2.txt", etc.)
paper_path = "../../tcm/collected_dataset/cleaned_papers_without_ref_new/"
print(os.path.exists(paper_path))

# Run the function to populate df['lda_topics']
df = populate_lda_topics(df, paper_path, n_topics=5, n_top_words=10, use_ngrams=True)

True
Topic #1: available, costs, analysis, scheduling, units, earliest, finish, latest, schedule, start
Topic #2: available, costs, analysis, scheduling, units, earliest, finish, latest, schedule, start
Topic #3: available, costs, analysis, scheduling, units, earliest, finish, latest, schedule, start
Topic #4: available, costs, analysis, scheduling, units, earliest, finish, latest, schedule, start
Topic #5: resource, activity, activities, construction, day, model, project, resources, time, critical
Topic #1: performance, project, management, construction, managers, competencies, model, team, competency, role
Topic #2: analysis, average, self, development, factors, factor, project management, range, project managers, key
Topic #3: analysis, average, self, development, factors, factor, project management, range, project managers, key
Topic #4: analysis, average, self, development, factors, factor, project management, range, project managers, key
Topic #5: analysis, average, self, develop

In [15]:
# Now each row in df has a list of unique topic words in df['lda_topics']
df

,Index,File_Name,Paper_Name,Keywords,Topics,Journal_Name,Published_Year,Country,Continent,URL,...,1986,1985,1984,1983,1982,1981,1980,topics,ground_truth_topics,lda_topics
0,0,(ASCE)0733-9364(1986)112_3(346).pdf,RESOURCE MANAGEMENT IN CONSTRUCTION,[],"['Resource management', 'Construction industry...",Journal of construction engineering and manage...,1986,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],"['Resource planning', 'cost control', 'constru...","[analysis, earliest, resources, activities, ti..."
1,2,(ASCE)0742-597X(2005)21_1(2).pdf,Competency-Based Model for Predicting Construc...,['Human factors; Professional development; Pro...,"['Construction project management', 'Competenc...",Journal of Management in Engineering,2005,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],"['Competency-based performance prediction', 'c...","[role, competencies, analysis, key, developmen..."
2,3,(ASCE)0887-3801(2006)20_3(165).pdf,Multi-Agent Framework for General-Purpose Situ...,['Models; Simulation; Construction management;...,"['Multi-agent framework', 'Situational simulat...",Journal of Computing in Civil Engineering,2006,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],"['Multi-agent systems', 'situational simulatio...","[cm, events, operators, activities, time, vari..."
3,4,(ASCE)1532-6748(2001)1_2(17).pdf,Construction Management Practices Are Slowly C...,[],NaN,Leadership and Management in Engineering,2001,NaN,4,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],['Strategic management in construction industr...,"[500, strategic management, based, management ..."
4,5,(ASCE)CO.1943-7862.0000100.pdf,Managerial Competencies of Female and Male Con...,['Women; Discrimination; Workplace diversity; ...,"['Female project managers', 'Managerial compet...",Journal of Construction Engineering and Manage...,2009,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],"['Gender representation in construction', 'Man...","[competencies, focus, industry, 2009, manageri..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
647,743,Texto.11.ConstructionQuality.pdf,Construction Quality Management: Principles an...,"['Manufacturing', 'Marketing', 'R&D and Engine...",NaN,NaN,2012,NaN,743,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],"['Quality Management', 'Total Quality Manageme...","[learning, implementation, business, activitie..."
648,744,Understanding the early stages of the innovati...,Understanding the early stages of the innovati...,"['Communication', 'critical perspective', 'dif...","['awareness', 'influence', 'communication netw...",Construction Management and Economics,2011,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],"['Innovation diffusion', 'awareness', 'influen...","[network, perspective, research, construction ..."
649,745,Use of attitude congruence to identify safety ...,Use of attitude congruence to identify safety ...,"['Safety', 'small business.']","['Attitude congruence', 'Safety interventions'...",Construction Management and Economics,2011,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],"['Construction industry safety', 'Occupational...","[attitudes, safety attitude, interventions, sa..."
650,746,Valuation of the minimum revenue guarantee and...,Valuation of the minimum revenue guarantee and...,"['BOT', 'infrastructure', 'real option', 'opti...","['Valuation', 'Minimum revenue guarantee', 'Op...",Construction Management and Economics,2006,China,Asia,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,[],"['Build-Operate-Transfer', 'minimum revenue gu...","[concessionaire, options, value, mrg, pre, val..."


In [16]:
# save the dataframe back to xlsx file
df.to_excel(os.path.join(folder_path, 'topics_citation_full_dataset_gl.xlsx'), index=False)